<a href="https://colab.research.google.com/github/liangliang6v6/Homeworks/blob/pages/Homework5_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Task 3 (55 points): NLP and Attention Mechanism
### Part 1 (10 points):
Implement the scaled dot-product attention from scratch (use NumPy and pandas only, no deep learning libraries are allowed for this step).

In [ ]:
import numpy as np
import pandas as pd

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: Query matrix of shape (batch_size, num_heads, seq_len, d_k)
    K: Key matrix of shape (batch_size, num_heads, seq_len, d_k)
    V: Value matrix of shape (batch_size, num_heads, seq_len, d_v)
    mask: Optional mask of shape (batch_size, 1, seq_len, seq_len)

    Returns:
    Output of shape (batch_size, num_heads, seq_len, d_v)
    """
    d_k = Q.shape[-1]

    scores = np.matmul(Q, K.transpose(0, 1, 3, 2)) / np.sqrt(d_k)

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)

    attention_weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
    attention_weights /= attention_weights.sum(axis=-1, keepdims=True)

    output = np.matmul(attention_weights, V)

    return output, attention_weights

np.random.seed(42)
batch_size = 2
num_heads = 1
seq_len = 4
d_k = 8
d_v = 8

Q = np.random.rand(batch_size, num_heads, seq_len, d_k)
K = np.random.rand(batch_size, num_heads, seq_len, d_k)
V = np.random.rand(batch_size, num_heads, seq_len, d_v)

# mask (e.g., for padding)
mask = np.ones((batch_size, 1, seq_len, seq_len))

output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)

# results using pandas
print("Output:\n", pd.DataFrame(output[0][0]))
print("\nAttention Weights:\n", pd.DataFrame(attention_weights[0][0]))


Output:
           0         1         2         3         4         5         6  \
0  0.246487  0.435795  0.599618  0.493692  0.466039  0.410585  0.634513   
1  0.237839  0.450471  0.616541  0.478096  0.484254  0.430922  0.610406   
2  0.245423  0.438116  0.605850  0.490584  0.471818  0.416615  0.626571   
3  0.240648  0.439304  0.610381  0.482694  0.470319  0.421662  0.623962   

          7  
0  0.401579  
1  0.421495  
2  0.407385  
3  0.411877  

Attention Weights:
           0         1         2         3
0  0.226730  0.260906  0.252353  0.260011
1  0.230054  0.252558  0.216142  0.301247
2  0.222275  0.258789  0.246016  0.272921
3  0.227691  0.250387  0.239681  0.282241


### Part 2 (10 points):
Pick any encoder-decoder seq2seq model and integrate the scaled dot-product attention in the encoder architecture. You may come
up with your own technique of integration or adopt one from literature. Hint: See Bahdanau or Luong attention paper.

In [ ]:
import numpy as np
import pandas as pd

class Seq2SeqAttention:
    def __init__(self, input_dim, hidden_dim, output_dim):
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim

        # Encoder and Decoder weights
        self.W_enc = np.random.randn(input_dim, hidden_dim) * 0.01
        self.W_dec = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.W_out = np.random.randn(hidden_dim, output_dim) * 0.01
    # in part 1
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        d_k = Q.shape[-1]
        scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(d_k)

        if mask is not None:
            scores = np.where(mask == 0, -1e9, scores)

        attention_weights = np.exp(scores - np.max(scores, axis=-1, keepdims=True))
        attention_weights /= attention_weights.sum(axis=-1, keepdims=True)

        output = np.matmul(attention_weights, V)
        return output, attention_weights

    def encoder(self, inputs):
        batch_size, seq_len, _ = inputs.shape
        hidden_states = np.tanh(np.matmul(inputs, self.W_enc))
        return hidden_states

    def decoder(self, encoder_outputs, target, mask=None):
        batch_size, seq_len, hidden_dim = encoder_outputs.shape
        hidden_state = np.zeros((batch_size, hidden_dim))
        outputs = []
        attention_weights_all = []

        for t in range(target.shape[1]):
            Q = hidden_state[:, None, :]
            K = encoder_outputs
            V = encoder_outputs

            context, attention_weights = self.scaled_dot_product_attention(Q, K, V, mask)
            attention_weights_all.append(attention_weights)

            hidden_state = np.tanh(np.matmul(context[:, 0, :], self.W_dec) + hidden_state)

            output = np.matmul(hidden_state, self.W_out)
            outputs.append(output)

        outputs = np.stack(outputs, axis=1)
        attention_weights_all = np.stack(attention_weights_all, axis=1)
        return outputs, attention_weights_all


    def forward(self, inputs, target, mask=None):
        encoder_outputs = self.encoder(inputs)
        outputs, attention_weights = self.decoder(encoder_outputs, target, mask)
        return outputs, attention_weights

np.random.seed(42)

batch_size = 2
input_dim = 5
hidden_dim = 8
output_dim = 10
seq_len = 6
target_len = 4

# Random input and target data
inputs = np.random.randn(batch_size, seq_len, input_dim)
target = np.random.randn(batch_size, target_len, output_dim)
mask = np.ones((batch_size, seq_len))

model = Seq2SeqAttention(input_dim, hidden_dim, output_dim)

outputs, attention_weights = model.forward(inputs, target, mask)

print("\nOutputs:\n", pd.DataFrame(outputs[0]))
print("\nAttention Weights:\n", pd.DataFrame(attention_weights[0, 0]))



Outputs:
           0         1         2         3         4             5         6  \
0 -0.000017  0.000002 -0.000012 -0.000004  0.000004  2.156776e-07 -0.000004   
1 -0.000034  0.000005 -0.000023 -0.000008  0.000007  4.313569e-07 -0.000009   
2 -0.000051  0.000007 -0.000035 -0.000013  0.000011  6.470423e-07 -0.000013   
3 -0.000067  0.000010 -0.000047 -0.000017  0.000015  8.627402e-07 -0.000017   

          7         8         9  
0  0.000006  0.000003 -0.000009  
1  0.000012  0.000005 -0.000018  
2  0.000018  0.000008 -0.000027  
3  0.000025  0.000010 -0.000036  

Attention Weights:
           0         1         2         3         4         5
0  0.166667  0.166667  0.166667  0.166667  0.166667  0.166667
1  0.166667  0.166667  0.166667  0.166667  0.166667  0.166667


###Part 3 (5 points):
Pick any public dataset of your choice (use a small-scale dataset like a
subset of the Tatoeba or Multi30k dataset) for machine translation task. Train your
model from Part 2 for the machine translation task. Evaluate test set by reporting the
BLEU Score.

In [ ]:
!pip3 install torchtext nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 61.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
!pip3 uninstall torch torchtext -y

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchtext 0.18.0
Uninstalling torchtext-0.18.0:
  Successfully uninstalled torchtext-0.18.0


In [5]:
!pip3 install torch torchvision torchtext --force-reinstall

  Using cached torchtext-0.18.0-cp311-cp311-manylinux1_x86_64.whl.metadata (7.9 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.4.127-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.1.0.70-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.4.5.8-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.2.1.3-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_curand_cu12-10.3.5.147-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cusolver_cu12-11.6.1.9-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cusparse_cu12-12.3.1.170-py3-none-manylinux2014_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nvjitlink_cu12-12.4

In [6]:
import torch
print("Torch version:", torch.__version__)
import torchtext
print("Torchtext version:", torchtext.__version__)
from torchtext.data.utils import get_tokenizer
from torchtext.datasets import Multi30k
from torchtext.vocab import build_vocab_from_iterator
from collections import Counter
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

class Seq2SeqAttention(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(Seq2SeqAttention, self).__init__()
        self.embedding = nn.Embedding(input_dim, hidden_dim)
        self.encoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, output_dim)

    def scaled_dot_product_attention(self, Q, K, V):
        d_k = Q.size(-1)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)
        return context, attention_weights

    def forward(self, src, tgt):
        # Encoder
        src_embedded = self.embedding(src)
        encoder_outputs, (hidden, cell) = self.encoder(src_embedded)

        # Decoder
        tgt_embedded = self.embedding(tgt)
        outputs = []
        attentions = []
        for t in range(tgt.size(1)):
            context, attn = self.scaled_dot_product_attention(hidden.transpose(0, 1), encoder_outputs, encoder_outputs)
            attentions.append(attn)
            output, (hidden, cell) = self.decoder(tgt_embedded[:, t:t+1, :], (context.transpose(0, 1), cell))
            outputs.append(self.out(output))

        outputs = torch.cat(outputs, dim=1)
        attentions = torch.cat(attentions, dim=1)
        return outputs, attentions


src_tokenizer = get_tokenizer('spacy', language='de_core_news_sm')
tgt_tokenizer = get_tokenizer('spacy', language='en_core_web_sm')

def yield_tokens(data_iter, tokenizer):
    for src, tgt in data_iter:
        yield tokenizer(src)
        yield tokenizer(tgt)

train_iter = Multi30k(split='train')

src_vocab = build_vocab_from_iterator(yield_tokens(train_iter, src_tokenizer), specials=['<unk>', '<pad>', '<bos>', '<eos>'])
src_vocab.set_default_index(src_vocab['<unk>'])

tgt_vocab = build_vocab_from_iterator(yield_tokens(train_iter, tgt_tokenizer), specials=['<unk>', '<pad>', '<bos>', '<eos>'])
tgt_vocab.set_default_index(tgt_vocab['<unk>'])

def text_to_tensor(text, vocab, tokenizer):
    tokens = [vocab['<bos>']] + [vocab[token] for token in tokenizer(text)] + [vocab['<eos>']]
    return torch.tensor(tokens, dtype=torch.long)

def pad_sequence(sequences, padding_value):
    max_len = max(len(seq) for seq in sequences)
    return torch.tensor([
        seq.tolist() + [padding_value] * (max_len - len(seq)) for seq in sequences
    ], dtype=torch.long)

BATCH_SIZE = 32

# use multi30k dataset
train_iter = Multi30k(split='train')
data = [(text_to_tensor(src, src_vocab, src_tokenizer),
         text_to_tensor(tgt, tgt_vocab, tgt_tokenizer)) for src, tgt in train_iter]


src_seqs = pad_sequence([x[0] for x in data], padding_value=src_vocab['<pad>'])
tgt_seqs = pad_sequence([x[1] for x in data], padding_value=tgt_vocab['<pad>'])

train_dataset = torch.utils.data.TensorDataset(src_seqs, tgt_seqs)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)



Torch version: 2.6.0+cu124


OSError: /usr/local/lib/python3.11/dist-packages/torchtext/lib/libtorchtext.so: undefined symbol: _ZN5torch3jit17parseSchemaOrNameERKSs

In [ ]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(tgt_vocab)
HIDDEN_DIM = 256

model = Seq2SeqAttention(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=src_vocab['<pad>'])

def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output, _ = model(src, tgt[:, :-1])
        loss = criterion(output.view(-1, OUTPUT_DIM), tgt[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# train for 10 epochs
EPOCHS = 10
for epoch in range(EPOCHS):
    loss = train(model, train_loader, optimizer, criterion)
    print(f'Epoch {epoch + 1}, Loss: {loss:.4f}')

# BLUE evaluate
def evaluate_bleu(model, loader):
    model.eval()
    references = []
    hypotheses = []
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            output, _ = model(src, tgt[:, :-1])
            output = torch.argmax(output, dim=-1).cpu().numpy()
            for i in range(output.shape[0]):
                pred_tokens = [tgt_vocab.lookup_token(t) for t in output[i] if t != tgt_vocab['<pad>']]
                true_tokens = [tgt_vocab.lookup_token(t) for t in tgt[i, 1:].cpu().numpy() if t != tgt_vocab['<pad>']]
                hypotheses.append(pred_tokens)
                references.append([true_tokens])

    bleu_score = corpus_bleu(references, hypotheses)
    return bleu_score

bleu_score = evaluate_bleu(model, train_loader)
print(f"BLEU Score: {bleu_score:.4f}")


###Part 4 (30 points):
In this part you are required to implement a simplified Transformer
model from scratch (using Python and NumPy/PyTorch/TensorFlow with minimal high-
level abstractions) and apply it to a machine translation task (e.g., English-to-French or
English-to-German translation) using the same dataset from part 3.

We discussed Transformer architecture in depth in class (Attention is
all you need). Apply the following simplifications to the original model architecture:
1. Reduced Model Depth: Use 2 encoder layers and 2 decoder layers instead of
the standard 6.
2. Limited Attention Heads: Use 2 attention heads in the multi-head attention
mechanism rather than 8.
3. Smaller Embedding Size: Set the embedding dimension to 64 instead of 512.
4. Reduced Feedforward Network Size: Use a feedforward dimension of 128
instead of 2048.
5. Smaller Dataset: Use a small dataset (e.g., about 10k sentence pairs).
6. Tokenization Simplifications: Use a basic subword tokenizer (like Byte Pair
Encoding - BPE) or word-level tokenization instead of complex language-specific
tokenizers.

Key components to implement:
1. Positional Encoding: Implement Sinusoidal position encoding.
2. Scaled dot-product attention: Use the same implementation from part 1.
3. Multi-Head Attention: Integrate the scaled dot-product attention into a multi-
head attention framework using the specified simplifications.
4. Encoder and Decoder Blocks: Implement simplified encoder and decoder
layers, ensuring: Layer normalization, Residual connections, Masked attention in
the decoder for autoregressive generation.
5. Final Output Layer: Implement a linear layer followed by a SoftMax activation
for generating translated tokens.

Evaluation: Compute the BLEU score on a validation set and compare the performance
with your model from part 2. Explain why there are differences in performance. Also
discuss any other differences you notice, for example runtime etc.